# Guía de Estudio QKP - Cuaderno 2: Resolución Clásica en PuLP y Stress Test de CPU

En este cuaderno programaremos la instancia del QKP que diseñamos a mano en el cuaderno 1. Aprenderás a linealizar los términos cuadráticos clásicamente usando **PuLP** y realizaremos un experimento de escalabilidad para ver cómo se "estresa" el procesador al aumentar el número de variables.

In [ ]:
# Instalar dependencias necesarias para este cuaderno
!pip install pulp matplotlib numpy

## 1. Implementación de nuestra Instancia QKP (4 variables) en PuLP

Para modelar un término cuadrático $x_i \cdot x_j$ en programación lineal entera (ILP), debemos realizar una **linealización**:
1. Introducimos una nueva variable binaria continua o entera: $z_{ij} \in \{0, 1\}$.
2. Añadimos tres restricciones lógicas para forzar que $z_{ij} = x_i \cdot x_j$:
   * $z_{ij} \le x_i$
   * $z_{ij} \le x_j$
   * $z_{ij} \ge x_i + x_j - 1$

Programemos esto:

In [ ]:
import pulp

# 1. Definir el problema de maximización
prob = pulp.LpProblem("QKP_Manual", pulp.LpMaximize)

# 2. Variables de decisión para los 4 objetos
x = {i: pulp.LpVariable(f"x_{i}", cat='Binary') for i in [1, 2, 3, 4]}

# 3. Variables auxiliares linealizadas para los pares con sinergia: (1,2), (2,3), (1,4)
synergy_pairs = [(1, 2), (2, 3), (1, 4)]
z = {pair: pulp.LpVariable(f"z_{pair[0]}_{pair[1]}", cat='Binary') for pair in synergy_pairs}

# 4. Función Objetivo con sinergias
# f(x) = 5*x1 + 6*x2 + 8*x3 + 4*x4 + 3*z_1_2 - 2*z_2_3 + 4*z_1_4
prob += (5*x[1] + 6*x[2] + 8*x[3] + 4*x[4] 
         + 3*z[(1,2)] - 2*z[(2,3)] + 4*z[(1,4)]), "Valor_Total"

# 5. Restricción de capacidad (Peso <= 5)
prob += (2*x[1] + 3*x[2] + 4*x[3] + 2*x[4] <= 5), "Capacidad_Mochila"

# 6. Restricciones de linealización para cada par de sinergia
for (i, j) in synergy_pairs:
    prob += z[(i, j)] <= x[i]
    prob += z[(i, j)] <= x[j]
    prob += z[(i, j)] >= x[i] + x[j] - 1

# 7. Resolver con el solver clásico por defecto (CBC)
status = prob.solve()

print(f"Estado de la solución: {pulp.LpStatus[status]}")
print("Objetos seleccionados:")
for i in [1, 2, 3, 4]:
    print(f"  Objeto {i}: {x[i].varValue}")
print(f"Valor total obtenido: {pulp.value(prob.objective)}")

## 2. El Experimento de Estrés (Benchmark de CPU)

¿Por qué decimos que el procesador clásico sufre con QKP? Vamos a generar instancias aleatorias de QKP de diferentes tamaños $N$. Para cada tamaño, habrá una densidad del 50% de sinergias aleatorias entre los objetos.

Ejecutaremos el solver clásico y graficaremos el tiempo de cálculo frente al número de variables para ver la curva exponencial.

In [ ]:
import time
import random
import numpy as np
import matplotlib.pyplot as plt

def solve_random_qkp(n, density=0.5):
    """Genera y resuelve una instancia aleatoria de QKP de tamaño n"""
    prob = pulp.LpProblem(f"QKP_{n}", pulp.LpMaximize)
    
    # Variables principales
    x = {i: pulp.LpVariable(f"x_{i}", cat='Binary') for i in range(n)}
    
    # Pesos y valores aleatorios
    weights = [random.randint(2, 10) for _ in range(n)]
    values = [random.randint(5, 20) for _ in range(n)]
    capacity = int(sum(weights) * 0.4)  # 40% del peso total como capacidad
    
    # Sinergias cuadráticas
    synergies = {}
    z = {}
    for i in range(n):
        for j in range(i+1, n):
            if random.random() < density:
                # Sinergia positiva (bono) o negativa (castigo)
                synergies[(i, j)] = random.randint(-5, 10)
                z[(i, j)] = pulp.LpVariable(f"z_{i}_{j}", cat='Binary')
                
    # Función objetivo
    obj_expr = pulp.lpSum(values[i] * x[i] for i in range(n)) 
    obj_expr += pulp.lpSum(synergies[pair] * z[pair] for pair in synergies)
    prob += obj_expr
    
    # Restricción de capacidad
    prob += pulp.lpSum(weights[i] * x[i] for i in range(n)) <= capacity
    
    # Restricciones de linealización
    for (i, j) in synergies:
        prob += z[(i, j)] <= x[i]
        prob += z[(i, j)] <= x[j]
        prob += z[(i, j)] >= x[i] + x[j] - 1
        
    # Resolver midiendo tiempo
    start_time = time.time()
    # Usamos msg=False para que no llene la consola con logs del solver
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    solve_time = time.time() - start_time
    
    num_total_vars = n + len(synergies)
    return solve_time, num_total_vars

In [ ]:
# Realizar el stress test para tamaños N de 5 a 60
sizes = [5, 10, 15, 20, 30, 40, 50, 60]
times = []
variables_count = []

print("Iniciando el benchmark clásico de QKP...")
for n in sizes:
    # Corremos 3 veces por tamaño para promediar el ruido de la CPU
    run_times = []
    for _ in range(3):
        t, num_vars = solve_random_qkp(n)
        run_times.append(t)
    avg_time = np.mean(run_times)
    times.append(avg_time)
    variables_count.append(num_vars)
    print(f"N = {n:2d} | Variables en PuLP = {num_vars:4d} | Tiempo medio = {avg_time:.4f} s")

In [ ]:
# Graficar los resultados del Stress Test
plt.figure(figsize=(10, 5))
plt.plot(sizes, times, 'o-', color='#e67e22', linewidth=2, label="Tiempo de Cómputo")
plt.title("Explosión Combinatoria en QKP Clásico (PuLP / CBC)", fontsize=14, fontweight='bold')
plt.xlabel("Número de Objetos Originales (N)", fontsize=12)
plt.ylabel("Tiempo de resolución (segundos)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

## 3. Conclusiones del Cuaderno clásico

Observa cómo la curva empieza plana pero rápidamente se empina (escala exponencial). Si intentáramos correr esto para $N=150$, la linealización clásica generaría más de **11,000 variables** y el tiempo de resolución podría dispararse a horas o días en tu laptop.

En el siguiente cuaderno, veremos cómo **QAOA** evita esta linealización inyectando la complejidad de las sinergias directamente como leyes físicas dentro del circuito cuántico.